# Week 3 Step 6: Why this customer was flagged

The churn model produces a probability. I do not ask the language model why. Python compares this customer with the typical customer who stayed, keeps the largest gaps, and only then asks the model to write those facts in plain language.

The decision threshold saved with the model is 0.46. A score of 0.61 or higher is high risk, 0.31 or lower is low risk, and anything between is borderline.

The first call scores `data/processed/customer_features.csv`. If that file is missing, it is rebuilt from the cleaned tables, which takes about a minute.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.explain_churn import build_facts, example_customers

examples = example_customers()
for label, customer_id in examples.items():
    facts = build_facts(customer_id)
    print(label)
    print(facts["facts"])
    print()


high
Customer CUST003961
Churn probability: 0.9901
Decision threshold: 0.46
Band: HIGH RISK
Current plan: Standard
Profile segment: Corporate Bundle
Model segment: Dormant at-risk
Tenure: 14 days

Risk factors (on the risky side of the typical customer who stayed):
- Failed-payment rate: 100.0% (typical non-churner: 6.7%). This customer is higher than typical. The model associates higher values with higher churn risk.
- Engagement score: 0.398 (typical non-churner: 0.552). This customer is lower than typical. The model associates lower values with higher churn risk.
- Login frequency (sessions per active month): 0.00 (typical non-churner: 1.50). This customer is lower than typical. The model associates lower values with higher churn risk.
- Tenure: 14 days (typical non-churner: 1419 days). This customer is lower than typical. The model associates lower values with higher churn risk.
- Lifetime value: 0.00 (typical non-churner: 201.37). This customer is lower than typical. The model ass

The next cell asks the model to explain each of those three customers. `numbers_outside_facts` lists any digit in the answer that is not already in the facts. An empty list means the write-up did not add a figure.


In [2]:
from src.llm.explain_churn import explain_churn, numbers_outside_facts

for label, customer_id in examples.items():
    result = explain_churn(customer_id)
    print(label, result["status"], result["band"], result["churn_probability"])
    print(result["answer"])
    print("numbers not in the facts:", numbers_outside_facts(result["answer"], result["facts"]))
    print()


high ok HIGH RISK 0.9901
This customer is flagged as high risk. The customer's churn probability is above the decision threshold.

This customer is flagged because their failed-payment rate is higher than typical, and the model associates higher values with higher churn risk. Their engagement score is lower than typical, and the model associates lower values with higher churn risk. Their login frequency is lower than typical, and the model associates lower values with higher churn risk. Their tenure is lower than typical, and the model associates lower values with higher churn risk. Their lifetime value is lower than typical, and the model associates lower values with higher churn risk.

The customer's average watch minutes per session is lower than typical, and the model associates higher values with higher churn risk. Their days since last viewing is lower than typical, and the model associates higher values with higher churn risk.

Consider reviewing this customer's account details.